In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/retrieval-rag/rag-from-scratch/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 06 · Iterative RAG (advanced)

Single-shot retrieve-then-read fails on **multi-hop** questions, where the
answer lives in two places and the second query only makes sense after you've
read the first result. Example: *"Which VPN client does the security policy
require, and which OSes does it support?"* — the client name is in one doc, the
OS list in another.

The fix is a loop: retrieve → decide what's still missing → retrieve again →
stop when you have enough (or hit a budget).


In [ ]:
# --- setup: make `import ragkit` work from notebooks/ or solutions/ ---
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
from ragkit.corpus import load_documents, load_corpus, load_qrels, tokenize
from ragkit.embed import get_embedder
from ragkit import llm

In [ ]:
from ragkit.reference import structure_aware_chunks, chunk_corpus, to_doc_ranking
docs = load_documents()
chunks = chunk_corpus(docs, structure_aware_chunks)
emb = get_embedder()
mat = emb.encode([c.text for c in chunks])

def retrieve(query, k=3):
    order = np.argsort(-(mat @ emb.encode(query)))[:k]
    return [chunks[i] for i in order]

# Show the failure: one retrieval for a 2-hop question rarely covers both docs.
q = next(x for x in load_qrels() if x["qid"] == "M1")
single = to_doc_ranking([c.chunk_id for c in retrieve(q["question"], k=3)])
print("question :", q["question"])
print("gold docs:", q["gold_docs"])
print("single-shot retrieved:", single)

### The planner

Deciding the next sub-query is the one step that wants a real LLM. So the
planner is pluggable: with an API key it asks the model to decompose the
question; offline it uses a hand-written decomposition for the demo questions,
so the loop still runs. (This mirrors production: swap the stub for the model.)


In [ ]:
# Hand-written decompositions so the notebook runs with no API key.
PLANS = {
  "M1": ["approved VPN client required by security policy SEC-011",
         "Meridian Connect supported operating systems macOS Windows Ubuntu"],
  "M2": ["who is paged for a SEV-1 customer-facing outage and how quickly",
         "how is production access granted approval on-call security lead"],
  "M3": ["Scale tier requests per minute rate limit",
         "Scale plan monthly price in SGD"],
}

def plan(question, qid=None):
    if llm.available().startswith(("anthropic", "openai")):
        prompt = ("Break this question into 2 short search queries, one per line, "
                  "each retrieving a different fact needed to answer it.\n\nQ: " + question)
        out = llm.complete(prompt, query=question)
        subs = [ln.strip("-- 	").strip() for ln in out.splitlines() if ln.strip()]
        if len(subs) >= 2:
            return subs[:3]
    return PLANS.get(qid, [question])

### Exercise — the iterative loop

Implement `iterative_retrieve`: walk the planned sub-queries, retrieve `k`
chunks for each, accumulate **unique** chunks (dedupe by `chunk_id`), and stop
once you hit `budget` sub-queries. Then answer over everything gathered.


In [ ]:
def iterative_retrieve(question, qid=None, k=3, budget=3):
    subqueries = plan(question, qid)
    gathered, seen, rounds = [], set(), 0
    for sub in subqueries:
        if rounds >= budget:
            break
        rounds += 1
        for c in retrieve(sub, k):
            if c.chunk_id not in seen:
                seen.add(c.chunk_id)
                gathered.append(c)
    return gathered, rounds

gathered, rounds = iterative_retrieve(q["question"], qid="M1", k=3, budget=3)
ids = [c.chunk_id for c in gathered]
assert len(ids) == len(set(ids))          # no duplicates
assert rounds <= 3                         # budget respected
covered = set(to_doc_ranking(ids))
print("rounds:", rounds, "| docs covered:", covered)
assert set(q["gold_docs"]) <= covered      # both hops found (was missed single-shot)

### Answer over the gathered evidence


In [ ]:
def build_prompt(query, gathered):
    blocks = [f"[{i}] (source: {c.doc_id})\n{c.text}" for i, c in enumerate(gathered, 1)]
    ctx = "\n\n".join(blocks)
    system = ("Answer using ONLY the sources. Cite [n] after each claim. "
              "The question has multiple parts — answer all of them.")
    return f"{system}\n\n=== SOURCES ===\n{ctx}\n\n=== QUESTION ===\n{query}", [c.text for c in gathered]

prompt, ctxs = build_prompt(q["question"], gathered)
print(llm.complete(prompt, contexts=ctxs, query=q["question"], embedder=emb))

**This is the seed of agentic RAG.** Make the planner a real model, let it
choose *which tool* to call (vector search, BM25, SQL, web) and *when to stop*,
add reflection on the results, and enforce a budget — and you have the loop that
powers modern "deep research" agents. The mechanics are exactly what you just
wrote; the sophistication is in the policy and the guardrails.

Where to go next: fine-tune the retriever/reranker on query logs, add a router
that sends simple questions down the cheap single-shot path, and evaluate
whole *trajectories* (did it find both docs? in how many steps?) rather than
just final answers. The primer's Part III and Part IV map the rest.
